In [1]:
"""
MDI vs. permutation-importance cross-check (Reviewer 2, Comment 3).


"""

import argparse
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import p rmutation_importance
from scipy.stats import spearmanr

SEED = 42
N_REPEATS = 5  # set by --n_repeats


def invert_normalize(score):
    inverted = score.max() - score
    if inverted.sum() > 0:
        return inverted / inverted.sum()
    return np.zeros_like(inverted)


def out_degree(fr_matrix):
    theta = fr_matrix.values.mean()
    A = (fr_matrix > theta).astype(int)
    return A.sum(axis=0)


def run_window(window_idx):
    path = f"log_window_{window_idx}.csv"
    if not os.path.exists(path):
        print(f"Skipping window {window_idx}: {path} not found in this folder.")
        return None

    df = pd.read_csv(path, index_col=0)
    n = len(df) - 1
    split = int(n * 0.8)

    print(f"\n{'='*70}\nWindow {window_idx}: {path}  (train={split}, test={n - split})\n{'='*70}")

    mdi_rows, perm_rows, oob_r2 = {}, {}, {}

    for company in df.columns:
        y = df[company].iloc[1:].reset_index(drop=True)
        x = df.drop(df.index[-1]).reset_index(drop=True)

        # Full-data model, exactly as in the paper (MDI ranking + free OOB score)
        model_full = RandomForestRegressor(n_estimators=200, max_features='log2',
                                            random_state=SEED, bootstrap=True,
                                            oob_score=True, n_jobs=-1)
        model_full.fit(x, y)
        mdi_rows[company] = invert_normalize(model_full.feature_importances_)
        oob_r2[company] = model_full.oob_score_

        # Chronological train/test split -> permutation importance on held-out data
        x_train, x_test = x.iloc[:split], x.iloc[split:]
        y_train, y_test = y.iloc[:split], y.iloc[split:]
        model_tt = RandomForestRegressor(n_estimators=200, max_features='log2',
                                          random_state=SEED, n_jobs=-1)
        model_tt.fit(x_train, y_train)
        perm = permutation_importance(model_tt, x_test, y_test, n_repeats=N_REPEATS,
                                       random_state=SEED, n_jobs=-1)
        perm_rows[company] = invert_normalize(np.clip(perm.importances_mean, 0, None))

    mdi_fr = pd.DataFrame(list(mdi_rows.values()), index=df.columns, columns=df.columns)
    perm_fr = pd.DataFrame(list(perm_rows.values()), index=df.columns, columns=df.columns)

    row_rhos = [spearmanr(mdi_fr.loc[c], perm_fr.loc[c])[0] for c in df.columns]
    row_rhos = np.array([r for r in row_rhos if not np.isnan(r)])

    degree_rho, degree_p = spearmanr(out_degree(mdi_fr), out_degree(perm_fr))
    r2_vals = np.array(list(oob_r2.values()))

    print(f"MDI vs. out-of-sample permutation importance:")
    print(f"  row-wise rho: mean = {row_rhos.mean():.3f}, min = {row_rhos.min():.3f}")
    print(f"  network out-degree (influence ranking) rho: {degree_rho:.3f} (p = {degree_p:.2e})")
    print(f"OOB R^2 (free bonus): mean = {r2_vals.mean():.3f}, "
          f"share > 0 = {(r2_vals > 0).mean():.1%}")

    return {"window": window_idx, "row_rho_mean": row_rhos.mean(),
            "degree_rho": degree_rho, "degree_p": degree_p,
            "oob_r2_mean": r2_vals.mean(), "oob_r2_pos_share": (r2_vals > 0).mean()}


def main():
    parser = argparse.ArgumentParser(description="MDI vs. permutation-importance cross-check")
    parser.add_argument("--window", type=int, default=None,
                         help="Window number 1-12. Omit to run all windows.")
    parser.add_argument("--n_repeats", type=int, default=5,
                         help="Permutation-importance repeats per company. Default 5.")
    args, _unknown = parser.parse_known_args()

    global N_REPEATS
    N_REPEATS = args.n_repeats

    windows = [args.window] if args.window else list(range(1, 13))

    summary = [r for w in windows if (r := run_window(w))]

    if summary:
        print(f"\n{'='*70}\nOVERALL SUMMARY (across {len(summary)} windows)\n{'='*70}")
        s = pd.DataFrame(summary)
        print(s.to_string(index=False))
        print(f"\nAverage row-wise rho: {s['row_rho_mean'].mean():.3f}")
        print(f"Average network out-degree rho: {s['degree_rho'].mean():.3f}")
        print(f"Average OOB R^2: {s['oob_r2_mean'].mean():.3f}")
        s.to_csv("mdi_vs_permutation_summary.csv", index=False)
        print("\nSaved mdi_vs_permutation_summary.csv")


if __name__ == "__main__":
    main()


C:\Users\Didar\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Didar\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



Window 1: log_window_1.csv  (train=81, test=21)


C:\Users\Didar\AppData\Local\Temp\ipykernel_17724\982878417.py:96: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  row_rhos = [spearmanr(mdi_fr.loc[c], perm_fr.loc[c])[0] for c in df.columns]


MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.115, min = -0.283
  network out-degree (influence ranking) rho: -0.002 (p = 9.89e-01)
OOB R^2 (free bonus): mean = -0.071, share > 0 = 15.4%

Window 2: log_window_2.csv  (train=81, test=21)
MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.169, min = -0.090
  network out-degree (influence ranking) rho: 0.227 (p = 1.64e-01)
OOB R^2 (free bonus): mean = -0.083, share > 0 = 7.7%

Window 3: log_window_3.csv  (train=81, test=21)
MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.180, min = -0.403
  network out-degree (influence ranking) rho: 0.284 (p = 8.00e-02)
OOB R^2 (free bonus): mean = -0.067, share > 0 = 7.7%

Window 4: log_window_4.csv  (train=81, test=21)
MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.190, min = -0.341
  network out-degree (influence ranking) rho: 0.235 (p = 1.50e-01)
OOB R^2 (free bonus): mean = -0.087, share > 0 = 7.7%

Window 5: 

C:\Users\Didar\AppData\Local\Temp\ipykernel_17724\982878417.py:96: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  row_rhos = [spearmanr(mdi_fr.loc[c], perm_fr.loc[c])[0] for c in df.columns]


MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.211, min = -0.233
  network out-degree (influence ranking) rho: 0.335 (p = 3.74e-02)
OOB R^2 (free bonus): mean = -0.054, share > 0 = 17.9%

Window 6: log_window_6.csv  (train=81, test=21)
MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.161, min = -0.311
  network out-degree (influence ranking) rho: 0.007 (p = 9.66e-01)
OOB R^2 (free bonus): mean = -0.091, share > 0 = 2.6%

Window 7: log_window_7.csv  (train=81, test=21)
MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.212, min = -0.148
  network out-degree (influence ranking) rho: 0.153 (p = 3.52e-01)
OOB R^2 (free bonus): mean = -0.072, share > 0 = 12.8%

Window 8: log_window_8.csv  (train=80, test=21)
MDI vs. out-of-sample permutation importance:
  row-wise rho: mean = 0.166, min = -0.253
  network out-degree (influence ranking) rho: 0.167 (p = 3.11e-01)
OOB R^2 (free bonus): mean = -0.085, share > 0 = 5.1%

Window 9: 

In [3]:
"""
Pooled (12-window) MDI vs. permutation-importance test (Reviewer 2, Comment 3).

"""

import argparse
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

SEED = 42
N_REPEATS = 5


def invert_normalize(score):
    inverted = score.max() - score
    if inverted.sum() > 0:
        return inverted / inverted.sum()
    return np.zeros_like(inverted)


def out_degree(fr_matrix):
    theta = fr_matrix.values.mean()
    A = (fr_matrix > theta).astype(int)
    return A.sum(axis=0)


def build_matrices(window_idx):
    path = f"log_window_{window_idx}.csv"
    if not os.path.exists(path):
        print(f"Skipping window {window_idx}: {path} not found in this folder.")
        return None

    df = pd.read_csv(path, index_col=0)
    n = len(df) - 1
    split = int(n * 0.8)

    mdi_rows, perm_rows = {}, {}
    for company in df.columns:
        y = df[company].iloc[1:].reset_index(drop=True)
        x = df.drop(df.index[-1]).reset_index(drop=True)

        model_full = RandomForestRegressor(n_estimators=200, max_features='log2',
                                            random_state=SEED, n_jobs=-1)
        model_full.fit(x, y)
        mdi_rows[company] = invert_normalize(model_full.feature_importances_)

        x_train, x_test = x.iloc[:split], x.iloc[split:]
        y_train, y_test = y.iloc[:split], y.iloc[split:]
        model_tt = RandomForestRegressor(n_estimators=200, max_features='log2',
                                          random_state=SEED, n_jobs=-1)
        model_tt.fit(x_train, y_train)
        perm = permutation_importance(model_tt, x_test, y_test, n_repeats=N_REPEATS,
                                       random_state=SEED, n_jobs=-1)
        perm_rows[company] = invert_normalize(np.clip(perm.importances_mean, 0, None))

    mdi_fr = pd.DataFrame(list(mdi_rows.values()), index=df.columns, columns=df.columns)
    perm_fr = pd.DataFrame(list(perm_rows.values()), index=df.columns, columns=df.columns)
    print(f"  window {window_idx} done ({len(df.columns)} companies)")
    return mdi_fr, perm_fr


def main():
    parser = argparse.ArgumentParser(description="Pooled MDI vs. permutation-importance test")
    parser.add_argument("--window", type=int, default=None,
                         help="Window number 1-12. Omit to run all windows and pool.")
    parser.add_argument("--n_repeats", type=int, default=5)
    args, _unknown = parser.parse_known_args()

    global N_REPEATS
    N_REPEATS = args.n_repeats

    windows = [args.window] if args.window else list(range(1, 13))

    pooled_mdi_deg_rank, pooled_perm_deg_rank = [], []
    pooled_mdi_row_rank, pooled_perm_row_rank = [], []

    print("Fitting models per window (this is the slow part)...")
    for w in windows:
        result = build_matrices(w)
        if result is None:
            continue
        mdi_fr, perm_fr = result

        # [1] network out-degree, ranked WITHIN this window before pooling
        mdi_deg = out_degree(mdi_fr)
        perm_deg = out_degree(perm_fr)
        pooled_mdi_deg_rank.extend(mdi_deg.rank().values)
        pooled_perm_deg_rank.extend(perm_deg.rank().values)

        # [2] row-wise, ranked WITHIN each company's row before pooling
        for company in mdi_fr.index:
            pooled_mdi_row_rank.extend(mdi_fr.loc[company].rank().values)
            pooled_perm_row_rank.extend(perm_fr.loc[company].rank().values)

    if not pooled_mdi_deg_rank:
        print("No windows found.")
        return

    rho_net, p_net = spearmanr(pooled_mdi_deg_rank, pooled_perm_deg_rank)
    rho_row, p_row = spearmanr(pooled_mdi_row_rank, pooled_perm_row_rank)

    print(f"\n{'='*70}\nPOOLED RESULTS (within-window/within-row standardized before pooling)\n{'='*70}")
    print(f"[1] Network out-degree (influence ranking), pooled across windows:")
    print(f"    n = {len(pooled_mdi_deg_rank)}, rho = {rho_net:.3f}, p = {p_net:.2e}")
    print(f"[2] Row-wise (predictor-level), pooled across all companies/windows:")
    print(f"    n = {len(pooled_mdi_row_rank)}, rho = {rho_row:.3f}, p = {p_row:.2e}")


if __name__ == "__main__":
    main()

Fitting models per window (this is the slow part)...
  window 1 done (39 companies)
  window 2 done (39 companies)
  window 3 done (39 companies)
  window 4 done (39 companies)
  window 5 done (39 companies)
  window 6 done (39 companies)
  window 7 done (39 companies)
  window 8 done (39 companies)
  window 9 done (39 companies)
  window 10 done (39 companies)
  window 11 done (39 companies)
  window 12 done (39 companies)

POOLED RESULTS (within-window/within-row standardized before pooling)
[1] Network out-degree (influence ranking), pooled across windows:
    n = 468, rho = 0.181, p = 8.17e-05
[2] Row-wise (predictor-level), pooled across all companies/windows:
    n = 18252, rho = 0.160, p = 2.86e-105
